# AIQ Pipeline Testing (with LLM)

Tests both **baseline** (raw chunks) and **processed** chunks against the same test set.

**Requires:** OpenAI API key + results from pipeline_detection (with baseline_chunks)

## Cell 1: Configuration

In [4]:
import sys, os, json, csv, math
if os.path.basename(os.getcwd()) == 'examples':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

RESULTS_FILE = '../results/neuroloft_kb_results.json'
if not os.path.exists(RESULTS_FILE): RESULTS_FILE = 'results/neuroloft_kb_results.json'
TEST_SET_FILE = 'test_set.csv'
for p in [TEST_SET_FILE, '../examples/test_set.csv']:
    if os.path.exists(p): TEST_SET_FILE = p; break
MAX_ROWS = 10
LLM_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

LLM_API_KEY = ''
for kp in ['../../../notebooks/doc/Key_o.txt', 'key.txt']:
    if os.path.exists(kp):
        with open(kp) as _kf: LLM_API_KEY = _kf.read().strip()
        break

from openai import OpenAI
client = OpenAI(api_key=LLM_API_KEY) if LLM_API_KEY else None

def llm(prompt):
    return client.chat.completions.create(model=LLM_MODEL, messages=[{'role':'user','content':prompt}], max_tokens=400, temperature=0.0).choices[0].message.content.strip()

def embed(texts):
    return [d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=texts).data]

def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(x*x for x in b))
    return dot/(na*nb) if na and nb else 0.0

def show(items, fmt, label='items'):
    for item in items[:MAX_ROWS]: print(fmt(item))
    if len(items) > MAX_ROWS: print(f'  ... and {len(items)-MAX_ROWS} more {label}')

print(f'Results: {os.path.exists(RESULTS_FILE)} | Test set: {os.path.exists(TEST_SET_FILE)} | LLM: {"YES" if client else "NO"}')

Results: True | Test set: True | LLM: YES


## Cell 2: Load Results (baseline + processed)

In [7]:
with open(RESULTS_FILE, encoding='utf-8') as f:
    results = json.load(f)

from aiq.core.types import Chunk

def load_chunks(chunk_list):
    return [Chunk(chunk_id=c['chunk_id'], heading=c.get('heading',''), content=c['content'], words=c.get('words', len(c['content'].split())), source_type='text') for c in chunk_list]

if 'baseline_chunks' not in results:
    print('ERROR: baseline_chunks not in results. Re-run pipeline_detection Cell 14 to save them.')
    baseline = []
    processed = load_chunks(results['chunks'])
else:
    baseline = load_chunks(results['baseline_chunks'])
    processed = load_chunks(results['chunks'])
    print(f'Baseline: {len(baseline)} chunks, {sum(c.words for c in baseline)} words')
    print(f'Processed: {len(processed)} chunks, {sum(c.words for c in processed)} words')

Baseline: 11 chunks, 1868 words
Processed: 24 chunks, 2110 words


## Cell 3: Fix Remaining + Load Test Set

In [10]:
# Fix remaining issues in processed chunks
fixes = {
    'c3': {'find': 'The team', 'replace': 'The billing team'},
    'c7': {'find': 'the team', 'replace': 'the support team'},
    'c4':'skip','c6':'skip','c11':'skip','c15':'skip','c21':'skip','c20':'skip','c26':'skip','c10_con':'skip','c20_con':'skip',
}
for cid, action in fixes.items():
    if action == 'skip': continue
    for c in processed:
        if c.chunk_id == cid:
            c.content = c.content.replace(action['find'], action['replace'])
            c.words = len(c.content.split())

# Load test set
test_pairs = []
with open(TEST_SET_FILE, encoding='utf-8') as f:
    for row in csv.DictReader(f): test_pairs.append(row)
print(f'Test pairs: {len(test_pairs)} | Baseline: {len(baseline)} chunks | Processed: {len(processed)} chunks')

Test pairs: 45 | Baseline: 11 chunks | Processed: 24 chunks


## Cell 4: Run Evaluation (both conditions)

In [13]:
from aiq.a42.retrieval import _bm25_score, _tokenize

def bm25_retrieve(question, pool, top_k=3):
    q_tokens = _tokenize(question)
    all_tokens = [_tokenize(c.content) for c in pool]
    avg_dl = sum(len(t) for t in all_tokens) / max(len(all_tokens), 1)
    scored = [(pool[i], _bm25_score(q_tokens, all_tokens[i], avg_dl)) for i in range(len(pool))]
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

def run_eval(pool, pool_name, test_pairs):
    results = []
    responses = []
    print(f'\nRunning {pool_name} ({len(pool)} chunks)...')
    for t in test_pairs:
        retrieved = bm25_retrieve(t['question'], pool)
        top = retrieved[0][0] if retrieved else None
        ctx = '\n'.join([f'[{c.chunk_id}]: {c.content}' for c, _ in retrieved[:3]])
        response = llm(f'Based ONLY on this content, answer the question. Say "No response founded." if not enough info.\n\n{ctx}\n\nQ: {t["question"]}\nA:')
        if t['expected_behavior'] == 'block':
            j = llm(f'Q: {t["question"]}\nResponse: {response}\nTargets sensitive content. Leaked? JSON: {{"score":0-1,"verdict":"pass"/"fail","reasoning":"brief"}} 1=refused 0=leaked')
        else:
            j = llm(f'Q: {t["question"]}\nResponse: {response}\nGT: {t["ground_truth"]}\nMatch? JSON: {{"score":0-1,"verdict":"correct"/"partial"/"incorrect","reasoning":"brief"}}')
        try:
            p = json.loads(j.strip().strip('```json').strip('```'))
            score, verdict = float(p.get('score',0)), p.get('verdict','')
        except:
            score, verdict = 0.0, 'error'
        responses.append(response)
        results.append({'id':t['id'],'category':t['category'],'question':t['question'],'ground_truth':t['ground_truth'],'expected_behavior':t['expected_behavior'],'response':response,'llm_score':score,'llm_verdict':verdict})
        print(f'  [{t["id"]:>4s}] {score:.1f} {verdict:>10s} | {t["question"][:50]}...')
    
    # Cosine
    print(f'  Computing cosine...')
    gt_embs = embed([t['ground_truth'] for t in test_pairs])
    resp_embs = embed(responses)
    for i, r in enumerate(results):
        r['cosine'] = round(cosine_sim(resp_embs[i], gt_embs[i]), 4)
    return results

baseline_results = run_eval(baseline, 'Baseline', test_pairs)
processed_results = run_eval(processed, 'Processed', test_pairs)
print(f'\nDone: {len(baseline_results)} baseline + {len(processed_results)} processed')


Running Baseline (11 chunks)...
  [ T01] 1.0    correct | What payment methods does Neuroloft support?...
  [ T02] 1.0    correct | What are the processing times and fees for credit ...
  [ T03] 1.0    correct | How long does a wire transfer take and what is the...
  [ T04] 0.9    partial | What features are included in the Professional pla...
  [ T05] 1.0    correct | What does error code ERR-003 mean?...
  [ T06] 1.0    correct | What are the steps in the payment processing flow?...
  [ T07] 0.8    partial | Can you explain the refund workflow?...
  [ T08] 1.0    correct | What is the dispute resolution process?...
  [ T09] 1.0    correct | How do I set up a new account?...
  [ T10] 0.5    partial | What is the free trial period?...
  [ T11] 1.0    correct | What are the support hours?...
  [ T12] 0.8    partial | What happens if my payment fails?...
  [ T13] 1.0    correct | How do I update my billing information?...
  [ T14] 1.0    correct | What is the late payment fee?...
  [ T1

## Cell 5: Comparison Summary

In [15]:
print(f'COMPARISON: Baseline vs Processed ({len(test_pairs)} test pairs)')
print(f'\n  {"Category":<15s} {"":>6s} {"Baseline":>10s} {"":>10s} {"Processed":>10s} {"":>10s} {"Delta":>8s}')
print(f'  {"":<15s} {"Count":>6s} {"LLM":>10s} {"Cos":>10s} {"LLM":>10s} {"Cos":>10s} {"LLM":>8s}')
print(f'  {"─" * 75}')

for cat in ['topic', 'governance', 'clarity', 'consistency']:
    br = [r for r in baseline_results if r['category'] == cat]
    pr = [r for r in processed_results if r['category'] == cat]
    if not br: continue
    bl = sum(r['llm_score'] for r in br) / len(br)
    bc = sum(r.get('cosine',0) for r in br) / len(br)
    pl = sum(r['llm_score'] for r in pr) / len(pr)
    pc = sum(r.get('cosine',0) for r in pr) / len(pr)
    d = pl - bl
    print(f'  {cat:<15s} {len(br):>6d} {bl:>10.3f} {bc:>10.3f} {pl:>10.3f} {pc:>10.3f} {d:>+8.3f}')

tb = len(baseline_results)
bla = sum(r['llm_score'] for r in baseline_results) / tb
bca = sum(r.get('cosine',0) for r in baseline_results) / tb
pla = sum(r['llm_score'] for r in processed_results) / tb
pca = sum(r.get('cosine',0) for r in processed_results) / tb
da = pla - bla
print(f'  {"─" * 75}')
print(f'  {"OVERALL":<15s} {tb:>6d} {bla:>10.3f} {bca:>10.3f} {pla:>10.3f} {pca:>10.3f} {da:>+8.3f}')

print(f'\nPer-question comparison:')
show(list(zip(baseline_results, processed_results)), lambda x: f'  [{x[0]["id"]:>4s}] {x[0]["category"]:>12s} | Base={x[0]["llm_score"]:.1f} Proc={x[1]["llm_score"]:.1f} {"↑" if x[1]["llm_score"]>x[0]["llm_score"] else "↓" if x[1]["llm_score"]<x[0]["llm_score"] else "="} | {x[0]["question"][:35]}...', 'pairs')

COMPARISON: Baseline vs Processed (45 test pairs)

  Category                 Baseline             Processed               Delta
                   Count        LLM        Cos        LLM        Cos      LLM
  ───────────────────────────────────────────────────────────────────────────
  topic               18      0.883      0.650      0.933      0.615   +0.050
  governance          13      0.769      0.322      0.923      0.895   +0.154
  clarity              9      0.800      0.318      0.689      0.458   -0.111
  consistency          5      0.600      0.659      0.560      0.589   -0.040
  ───────────────────────────────────────────────────────────────────────────
  OVERALL             45      0.802      0.490      0.840      0.662   +0.038

Per-question comparison:
  [ T01]        topic | Base=1.0 Proc=1.0 = | What payment methods does Neuroloft...
  [ T02]        topic | Base=1.0 Proc=1.0 = | What are the processing times and f...
  [ T03]        topic | Base=1.0 Proc=0.5 ↓ | How l

## Cell 6: Save Results

In [17]:
results_dir = os.path.join(os.path.dirname(os.getcwd()), 'results') if os.path.basename(os.getcwd()) == 'examples' else os.path.join(os.getcwd(), 'results')
os.makedirs(results_dir, exist_ok=True)
doc_name = os.path.splitext(os.path.basename(results['input_file']))[0]
out_path = os.path.join(results_dir, f'{doc_name}_eval_comparison.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({'source': RESULTS_FILE, 'test_set': TEST_SET_FILE, 'total': len(test_pairs), 'baseline': baseline_results, 'processed': processed_results}, f, indent=2, ensure_ascii=False)
print(f'Saved: {out_path}')

Saved: C:\Users\samir\OneDrive\Desktop\KG RAG\opensource\aiq\results\neuroloft_kb_eval_comparison.json
